# Why is the Poultry Model Worse? Investigation

The poultry model (poultry vs everything else) has **inspected F1=0.64** compared to the baseline (farm vs not-farm) at **F1=0.90**. This notebook investigates why.

**Hypothesis**: Pig and cattle farms look identical to poultry farms from Sentinel-2 at 10m resolution. The model can distinguish farms from non-farms, but cannot reliably distinguish farm *types* from satellite imagery alone.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

# Paths — change ROOT if running locally vs on pod
ROOT = Path("/tmp/poultry_analysis")  # local
# ROOT = Path("/workspace/farm-mapping/data/output")  # pod

RACHEL_PATH = Path("data/rachel_geometry_candidates/selected_clusters_relabeled.parquet")
rachel = pd.read_parquet(RACHEL_PATH)
label_map = dict(zip(rachel['cluster_id'].astype(str), rachel['modified_label']))
adm0_map = dict(zip(rachel['cluster_id'].astype(str), rachel['ADM0']))
score_if_map = dict(zip(rachel['cluster_id'].astype(str), rachel['template_score_if']))
adm0_to_name = {'USA':'United States','BRA':'Brazil','MEX':'Mexico','THA':'Thailand','CHL':'Chile'}

COLORS = {'TP': '#2ecc71', 'FP': '#e74c3c', 'FN': '#f39c12', 'TN': '#3498db'}

def load_experiment(scored_path, splits_path):
    df = pd.read_parquet(scored_path)
    splits = pd.read_csv(splits_path)
    df['split'] = df['candidate_id'].astype(str).map(dict(zip(splits['candidate_id'].astype(str), splits['split'])))
    df['farm_type'] = df['candidate_id'].astype(str).map(label_map)
    df['country'] = df['candidate_id'].astype(str).map(adm0_map).map(adm0_to_name)
    df['template_score_if'] = df['candidate_id'].astype(str).map(score_if_map)
    df['pred_class'] = 'TN'
    df.loc[(df['predicted_label']==1)&(df['true_label']==1), 'pred_class'] = 'TP'
    df.loc[(df['predicted_label']==1)&(df['true_label']==0), 'pred_class'] = 'FP'
    df.loc[(df['predicted_label']==0)&(df['true_label']==1), 'pred_class'] = 'FN'
    return df

poultry = load_experiment(ROOT / 'scored_poultry.parquet', ROOT / 'splits_poultry.csv')
baseline = load_experiment(ROOT / 'scored_baseline.parquet', ROOT / 'splits_baseline.csv')

print(f'Poultry model: {len(poultry)} scored, inspected={len(poultry[poultry["split"]=="inspected"])}')
print(f'Baseline model: {len(baseline)} scored, inspected={len(baseline[baseline["split"]=="inspected"])}')

## 1. Head-to-Head Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, df) in zip(axes, [('Baseline (farm vs not-farm)', baseline), ('Poultry (poultry vs rest)', poultry)]):
    insp = df[df['split']=='inspected']
    counts = [insp['pred_class'].value_counts().get(k, 0) for k in ['TP','FP','FN','TN']]
    ax.bar(['TP','FP','FN','TN'], counts, color=[COLORS[k] for k in ['TP','FP','FN','TN']])
    ax.set_title(name)
    for i, v in enumerate(counts):
        ax.text(i, v+3, str(v), ha='center', fontsize=11)
    tp, fp, fn, tn = counts
    prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1); f1 = 2*prec*rec/max(prec+rec,1e-8)
    ax.set_xlabel(f'P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}', fontsize=12, fontweight='bold')

plt.suptitle('Inspected Set: Baseline vs Poultry Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Root Cause: What Gets Misclassified?

The poultry model's negatives include pig farms, cattle farms, and non-farms. If pig/cattle farms look like poultry from satellite, they'll be false positives.

In [ ]:
insp_p = poultry[poultry['split']=='inspected']

# Score distribution by farm type
farm_types = ['Farm: Poultry: Unspecified/Other', 'Farm: Pigs', 'Farm: Cattle',
              'Farm: PigsOrPoultry', 'NotFarm']
type_colors = {'Farm: Poultry: Unspecified/Other': '#2ecc71', 'Farm: Pigs': '#3498db',
               'Farm: Cattle': '#9b59b6', 'Farm: PigsOrPoultry': '#f39c12', 'NotFarm': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Score histogram by farm type
for ft in farm_types:
    sub = insp_p[insp_p['farm_type']==ft]
    if len(sub) == 0: continue
    axes[0].hist(sub['predicted_score'], bins=25, alpha=0.5,
                 label=f"{ft.replace('Farm: ','')} (n={len(sub)})",
                 color=type_colors.get(ft, 'gray'))
axes[0].axvline(0.5, color='black', linestyle='--', alpha=0.5, label='Threshold (0.5)')
axes[0].set_xlabel('Poultry Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Poultry Model Score by Farm Type (Inspected)')
axes[0].legend(fontsize=9)

# Boxplot
data_for_box = []
labels_for_box = []
colors_for_box = []
for ft in farm_types:
    sub = insp_p[insp_p['farm_type']==ft]
    if len(sub) == 0: continue
    data_for_box.append(sub['predicted_score'].values)
    labels_for_box.append(ft.replace('Farm: ','').replace('Unspecified/Other','Poultry')[:15])
    colors_for_box.append(type_colors.get(ft, 'gray'))

bp = axes[1].boxplot(data_for_box, labels=labels_for_box, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors_for_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].axhline(0.5, color='black', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Poultry Probability')
axes[1].set_title('Score Distribution by Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Key numbers
pigs = insp_p[insp_p['farm_type']=='Farm: Pigs']
cattle = insp_p[insp_p['farm_type']=='Farm: Cattle']
notfarm = insp_p[insp_p['farm_type']=='NotFarm']
print(f'Pig farms predicted as poultry: {(pigs["predicted_label"]==1).sum()}/{len(pigs)} ({100*(pigs["predicted_label"]==1).mean():.0f}%), median score={pigs["predicted_score"].median():.3f}')
print(f'Cattle farms predicted as poultry: {(cattle["predicted_label"]==1).sum()}/{len(cattle)} ({100*(cattle["predicted_label"]==1).mean():.0f}%), median score={cattle["predicted_score"].median():.3f}')
print(f'NotFarm predicted as poultry: {(notfarm["predicted_label"]==1).sum()}/{len(notfarm)} ({100*(notfarm["predicted_label"]==1).mean():.0f}%), median score={notfarm["predicted_score"].median():.3f}')

## 3. Pig Farms vs Poultry Farms — Side by Side

Compare the baseline model's view (both are farms → both score high) vs the poultry model's view (should separate them).

In [ ]:
# Merge both models' scores for the same candidates
merged = poultry[['candidate_id','predicted_score','farm_type','country']].rename(columns={'predicted_score':'poultry_score'})
baseline_scores = baseline[['candidate_id','predicted_score']].rename(columns={'predicted_score':'baseline_score'})
merged = merged.merge(baseline_scores, on='candidate_id', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: baseline score vs poultry score, colored by farm type
for ft in farm_types:
    sub = merged[merged['farm_type']==ft]
    if len(sub) == 0: continue
    short = ft.replace('Farm: ','').replace('Unspecified/Other','Poultry')[:15]
    axes[0].scatter(sub['baseline_score'], sub['poultry_score'], alpha=0.3, s=10,
                    label=f'{short} ({len(sub)})', color=type_colors.get(ft, 'gray'))
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Baseline Score (farm probability)')
axes[0].set_ylabel('Poultry Score (poultry probability)')
axes[0].set_title('Baseline vs Poultry Model Scores')
axes[0].legend(markerscale=3, fontsize=9)

# Just pigs vs poultry
pigs_m = merged[merged['farm_type']=='Farm: Pigs']
poultry_m = merged[merged['farm_type']=='Farm: Poultry: Unspecified/Other']
axes[1].hist(poultry_m['poultry_score'], bins=30, alpha=0.6, label=f'Poultry farms (n={len(poultry_m)})', color='#2ecc71')
axes[1].hist(pigs_m['poultry_score'], bins=30, alpha=0.6, label=f'Pig farms (n={len(pigs_m)})', color='#3498db')
axes[1].axvline(0.5, color='black', linestyle='--', alpha=0.5, label='Threshold')
axes[1].set_xlabel('Poultry Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Poultry vs Pig Farms — Model Cannot Separate Them')
axes[1].legend()

plt.tight_layout()
plt.show()

# Overlap metric
from sklearn.metrics import roc_auc_score
pig_poultry = merged[merged['farm_type'].isin(['Farm: Pigs','Farm: Poultry: Unspecified/Other'])].copy()
pig_poultry['is_poultry'] = (pig_poultry['farm_type'].str.contains('Poultry')).astype(int)
auc = roc_auc_score(pig_poultry['is_poultry'], pig_poultry['poultry_score'])
print(f'\nAUC for separating poultry from pigs: {auc:.3f} (0.5 = random, 1.0 = perfect)')

## 4. Country Breakdown — Where Does the Poultry Model Fail Most?

In [ ]:
insp_p = poultry[poultry['split']=='inspected']
insp_b = baseline[baseline['split']=='inspected']

print(f'{"Country":15s}  {"--- Baseline ---":>30s}  {"--- Poultry ---":>30s}')
print(f'{"":15s}  {"TP":>4s} {"FP":>4s} {"FN":>4s} {"TN":>4s} {"F1":>6s}  {"TP":>4s} {"FP":>4s} {"FN":>4s} {"TN":>4s} {"F1":>6s}')
print('-' * 85)

for country in sorted(insp_p['country'].dropna().unique()):
    # Baseline
    gb = insp_b[insp_b['country']==country]
    tp_b = (gb['pred_class']=='TP').sum()
    fp_b = (gb['pred_class']=='FP').sum()
    fn_b = (gb['pred_class']=='FN').sum()
    tn_b = (gb['pred_class']=='TN').sum()
    f1_b = 2*tp_b/(2*tp_b+fp_b+fn_b) if (2*tp_b+fp_b+fn_b)>0 else 0
    
    # Poultry
    gp = insp_p[insp_p['country']==country]
    tp_p = (gp['pred_class']=='TP').sum()
    fp_p = (gp['pred_class']=='FP').sum()
    fn_p = (gp['pred_class']=='FN').sum()
    tn_p = (gp['pred_class']=='TN').sum()
    f1_p = 2*tp_p/(2*tp_p+fp_p+fn_p) if (2*tp_p+fp_p+fn_p)>0 else 0
    
    print(f'{country:15s}  {tp_b:4d} {fp_b:4d} {fn_b:4d} {tn_b:4d} {f1_b:6.3f}  {tp_p:4d} {fp_p:4d} {fn_p:4d} {tn_p:4d} {f1_p:6.3f}')

## 5. FP Deep Dive: What Do Misclassified Pig Farms Look Like?

In [ ]:
# Pig farms that the poultry model wrongly calls poultry
# vs pig farms it correctly rejects
pigs_insp = insp_p[insp_p['farm_type']=='Farm: Pigs'].copy()
pigs_fp = pigs_insp[pigs_insp['pred_class']=='FP']  # wrongly called poultry
pigs_tn = pigs_insp[pigs_insp['pred_class']=='TN']  # correctly rejected

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(pigs_fp['predicted_score'], bins=20, alpha=0.6, color='#e74c3c', label=f'FP: called poultry (n={len(pigs_fp)})')
axes[0].hist(pigs_tn['predicted_score'], bins=20, alpha=0.6, color='#3498db', label=f'TN: correctly rejected (n={len(pigs_tn)})')
axes[0].axvline(0.5, color='black', linestyle='--')
axes[0].set_xlabel('Poultry Score')
axes[0].set_title('Pig Farms: FP vs TN Score Distribution')
axes[0].legend()

# By country
ct = pd.crosstab(pigs_insp['country'], pigs_insp['pred_class'])
ct[['FP','TN']].plot.bar(ax=axes[1], color=['#e74c3c','#3498db'])
axes[1].set_title('Pig Farms: FP vs TN by Country')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Template score comparison: do misclassified pigs have different morphology?
print(f'\nIsolation Forest score for pig farms:')
print(f'  FP (called poultry):    median={pigs_fp["template_score_if"].median():.3f}, mean={pigs_fp["template_score_if"].mean():.3f}')
print(f'  TN (correctly rejected): median={pigs_tn["template_score_if"].median():.3f}, mean={pigs_tn["template_score_if"].mean():.3f}')
print(f'\n=> Pig farms that look like poultry have {"higher" if pigs_fp["template_score_if"].median() > pigs_tn["template_score_if"].median() else "similar"} IF scores — they may have similar building morphology.')

## 6. Conclusion

In [ ]:
print("""
FINDINGS:
=========

1. MAIN ISSUE: Pig farms and poultry farms are nearly indistinguishable
   from Sentinel-2 imagery at 10m resolution.
   - 43% of pig farms are predicted as poultry (median score 0.65)
   - Both have similar long rectangular buildings, similar density
   - The CNN learns "farm-like" features, not species-specific features

2. NEGATIVE CLASS IS HARDER: The poultry model's negative class includes:
   - Pig farms (280 in inspected) — look like poultry farms
   - Cattle farms (94) — somewhat different but overlap
   - NotFarm (303) — easiest to separate (only 18% FP rate)
   
   The baseline model only needs to separate farms from non-farms,
   which is much easier.

3. RECALL DROPS TOO: 77% poultry recall vs 97% baseline recall
   - The model becomes more conservative overall
   - Some actual poultry farms score below threshold

4. IMPLICATION: Sentinel-2 at 10m is NOT sufficient for farm TYPE
   classification. It's good for farm DETECTION (farm vs not-farm)
   but not for distinguishing species. This would require:
   - Higher resolution imagery (sub-meter)
   - Building morphology features (from Rachel's pipeline)
   - Or a combined approach (CNN detects farms, morphology classifies type)
""")